# Vigenere - PATE training

Trains the main model: rotary position encoding, period conditioning through a
single learned vector per candidate length, and key-length and validity heads.

Checkpoints go to `checkpoints/vigenere_pate`.

In [ ]:
import os, sys
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/drive/MyDrive/cryptanalysis-ai')
    %pip install -q pytorch-lightning torchmetrics tensorboard matplotlib
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == 'notebooks':
        REPO_DIR = REPO_DIR.parent

os.environ['CRYPT_DATA_DIR'] = str(REPO_DIR / 'data')
os.environ['CRYPT_CKPT_DIR'] = str(REPO_DIR / 'checkpoints')
os.environ['CRYPT_LOG_DIR']  = str(REPO_DIR / 'tb_logs')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('repo:', REPO_DIR)


In [ ]:
import torch, config
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

from models.vigenere.model_pate import VigenereSolver
from models.vigenere.data_module import VigenereDataModule

print(config.describe())


In [ ]:
SEED = 42
pl.seed_everything(SEED, workers=True)

MIN_K_LEN, MAX_K_LEN, BATCH_SIZE = 3, 32, 128

dm = VigenereDataModule(
    text_path=str(config.data_file("final_dataset_shuffled.txt")),
    alphabet=config.ALPHABET,
    batch_size=BATCH_SIZE,
    min_key_len=MIN_K_LEN,
    max_key_len=MAX_K_LEN,
)
dm.setup()

model = VigenereSolver(
    vocab_size=dm.dataset.crypto_vocab_size,
    min_key_len=MIN_K_LEN, max_key_len=MAX_K_LEN,
    d_model=256, nhead=8, num_layers=6, lr=1e-3,
)

n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters: {n:,}")
print(f"training modes (correct/incorrect/blind): {model.CORRECT_KEY_PROB}/"
      f"{model.WRONG_KEY_PROB}/{model.UNKNOWN_KEY_PROB}")
print(f"loss weights (validity/key-length): {model.VALIDITY_WEIGHT}/{model.KEYLEN_WEIGHT}")
print(f"rope wavelengths: {[round(v,1) for v in model.rope_wavelengths.tolist()]}")


In [ ]:
MODEL_NAME = "vigenere_pate"
MAX_EPOCHS = 230
ckpt_dir, log_dir = config.model_paths(MODEL_NAME)
print("checkpoints ->", ckpt_dir)

callbacks = [
    ModelCheckpoint(
        dirpath=str(ckpt_dir),
        filename="pate-best-acc-{epoch:02d}-{val_acc:.3f}",
        save_top_k=2, save_last=True, monitor="val_acc", mode="max",
    ),
    ModelCheckpoint(
        dirpath=str(ckpt_dir),
        filename="pate-best-keylen-{epoch:02d}-{val_keylen_acc:.3f}-{val_acc:.3f}",
        save_top_k=2, monitor="val_keylen_acc", mode="max",
    ),
    EarlyStopping(monitor="val_keylen_acc", patience=40, mode="max", strict=False),
]

logger = TensorBoardLogger(str(log_dir), name=MODEL_NAME)
precision = "32-true" if torch.cuda.is_available() else "32-true"

trainer = pl.Trainer(
    accelerator="auto", devices=1,
    max_epochs=MAX_EPOCHS, min_epochs=80,
    callbacks=callbacks, logger=logger,
    gradient_clip_val=1.0, log_every_n_steps=10, precision=precision,
)

trainer.fit(model, datamodule=dm)
# to resume: trainer.fit(model, datamodule=dm, ckpt_path=str(ckpt_dir / "last.ckpt"))


Mixed precision is not used here: at the wider key range the key-length
cross-entropy over thirty classes overflowed in fp16 and produced NaN losses.